# Round 3 Market Analysis

This notebook loads **Round 3** price and trade data, then builds readable plots to answer:

- Which products are trending or mean-reverting?
- Which products have wide or unstable spreads?
- How do option-like products (`VEV_<strike>`) behave across strikes?
- Do trade prices track the quoted mid-price?


In [ ]:
%matplotlib inline
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "lines.linewidth": 1.6,
})

DATA_DIR = Path("Data/ROUND3")
DAYS = [0, 1, 2]

print("Setup complete.")


: 

In [ ]:
price_frames = []
trade_frames = []

for day in DAYS:
    p_file = DATA_DIR / f"prices_round_3_day_{day}.csv"
    t_file = DATA_DIR / f"trades_round_3_day_{day}.csv"

    p = pd.read_csv(p_file, sep=';')
    t = pd.read_csv(t_file, sep=';')
    t['day'] = day

    price_frames.append(p)
    trade_frames.append(t)

prices = pd.concat(price_frames, ignore_index=True)
trades = pd.concat(trade_frames, ignore_index=True)

prices['abs_ts'] = prices['day'] * 1_000_000 + prices['timestamp']
trades['abs_ts'] = trades['day'] * 1_000_000 + trades['timestamp']

# Standard microstructure features.
prices['spread_1'] = prices['ask_price_1'] - prices['bid_price_1']
prices['mid_ret'] = prices.groupby(['day', 'product'])['mid_price'].pct_change().fillna(0.0)
prices['imbalance_1'] = (prices['bid_volume_1'] - prices['ask_volume_1']) / (prices['bid_volume_1'] + prices['ask_volume_1']).replace(0, np.nan)

prices['strike'] = pd.to_numeric(prices['product'].str.extract(r'^VEV_(\d+)$')[0], errors='coerce')

print(f"Loaded {len(prices):,} price rows and {len(trades):,} trade rows.")
print(f"Products: {prices['product'].nunique()}")


In [ ]:
quality = (
    prices.groupby('product')
    .agg(
        rows=('product', 'size'),
        days=('day', 'nunique'),
        zero_mid=('mid_price', lambda s: int((s == 0).sum())),
        pct_zero_mid=('mid_price', lambda s: 100 * (s == 0).mean()),
        avg_spread=('spread_1', 'mean'),
    )
    .sort_values(['pct_zero_mid', 'rows'], ascending=[False, False])
)

quality.head(12)


In [ ]:
# Plot 1: Mid-price trajectories for the most active products.
top_products = prices['product'].value_counts().head(8).index.tolist()
fig, axes = plt.subplots(len(top_products), 1, figsize=(14, 2.5 * len(top_products)), sharex=True)

if len(top_products) == 1:
    axes = [axes]

day_colors = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c'}

for ax, product in zip(axes, top_products):
    sub = prices[prices['product'] == product]
    for day, d in sub.groupby('day'):
        d = d.sort_values('timestamp')
        ax.plot(d['timestamp'], d['mid_price'], color=day_colors.get(day, '#999999'), alpha=0.25, label=f'Day {day}')
        smooth = d['mid_price'].rolling(window=800, min_periods=1).mean()
        ax.plot(d['timestamp'], smooth, color=day_colors.get(day, '#999999'), alpha=0.95)

    ax.set_title(product)
    ax.set_ylabel('Mid')
    ax.legend(loc='upper left', ncol=3)

axes[-1].set_xlabel('Timestamp (within day)')
fig.suptitle('Round 3 Mid-Price by Product (raw + smoothed)', y=1.002, fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: Spread and realized volatility comparison for ranking products by trading quality.
summary = (
    prices.groupby('product')
    .agg(
        mean_spread=('spread_1', 'mean'),
        p90_spread=('spread_1', lambda s: np.nanpercentile(s, 90)),
        realized_vol_bp=('mid_ret', lambda s: np.nanstd(s) * 10_000),
    )
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

summary.sort_values('mean_spread').plot(kind='bar', y='mean_spread', ax=ax[0], color='#4c78a8', legend=False)
ax[0].set_title('Average Top-of-Book Spread by Product')
ax[0].set_ylabel('Spread (ask1 - bid1)')
ax[0].set_xlabel('')
ax[0].tick_params(axis='x', rotation=55)

summary.sort_values('realized_vol_bp').plot(kind='bar', y='realized_vol_bp', ax=ax[1], color='#f58518', legend=False)
ax[1].set_title('Realized Mid-Return Volatility by Product')
ax[1].set_ylabel('Volatility (basis points)')
ax[1].set_xlabel('')
ax[1].tick_params(axis='x', rotation=55)

plt.tight_layout()
plt.show()


In [ ]:
# Plot 3: Option-like surface view for VEV_<strike> products.
vev = prices.dropna(subset=['strike']).copy()
vev = vev[vev['mid_price'] > 0]

surface = (
    vev.groupby(['day', 'strike'])['mid_price']
    .median()
    .reset_index()
    .sort_values(['day', 'strike'])
)

fig, ax = plt.subplots(figsize=(12, 6))
palette = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c'}

for day, sub in surface.groupby('day'):
    ax.plot(sub['strike'], sub['mid_price'], marker='o', label=f'Day {day}', color=palette.get(day, '#888888'))

ax.set_title('VEV Mid-Price by Strike (Median per Day)')
ax.set_xlabel('Strike')
ax.set_ylabel('Median Mid-Price')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Plot 4: Trade prices overlaid on quoted mid for the most traded symbol.
most_traded = trades['symbol'].value_counts().idxmax()

p = prices[prices['product'] == most_traded].copy()
t = trades[trades['symbol'] == most_traded].copy()

fig, ax = plt.subplots(figsize=(14, 6))

for day, sub in p.groupby('day'):
    sub = sub.sort_values('timestamp')
    ax.plot(sub['timestamp'], sub['mid_price'], label=f'Mid Day {day}', alpha=0.85)

if not t.empty:
    marker_size = np.clip(t['quantity'].fillna(1).astype(float) * 4, 12, 180)
    ax.scatter(
        t['timestamp'] % 1_000_000,
        t['price'],
        s=marker_size,
        alpha=0.35,
        c='#d62728',
        edgecolors='none',
        label='Trades (size~quantity)'
    )

ax.set_title(f'{most_traded}: Quoted Mid vs Executed Trades')
ax.set_xlabel('Timestamp (within day)')
ax.set_ylabel('Price')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Most traded symbol in Round 3: {most_traded}')


## Notes

- If a product has many zero mid-prices, treat spread/volatility metrics cautiously.
- For strategy work, add rolling z-scores and lead/lag tests between `VELVETFRUIT_EXTRACT` and `VEV_*`.
- If you want signal prototyping next, this notebook can be extended into a backtest-ready feature table.
